# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: Cuarta Dimensión

* Ezequiel Espejo LU 483/22
* Melanie Espinoza LU 1635/23
* Jimena Jofré LU 696/23
* Alison Herrera 814/23

In [1]:
from alc import *
import numpy as np
import cv2

### Ejercicio 1 - DLT ecuaciones formales

### Ejercicio 1.a – Rango de Q y cantidad mínima de puntos

A partir de la ecuación (6) del enunciado, tenemos el sistema
homogéneo:

$$
Qm = 0
$$

donde $Q \in \mathbb{R}^{2n \times 12}$ y
$m \in \mathbb{R}^{12}$ contiene los coeficientes de
la matriz de proyección $M$.

Queremos determinar qué rango debe tener $Q$ para garantizar
una solución no nula única, salvo un factor de escala.

**1. Condición sobre el rango de Q**

Como el sistema es homogéneo, siempre admite la solución
trivial $m=0$. Sin embargo, buscamos una solución no nula
que represente una matriz de proyección.

Por el teorema de rango-nulidad:

$$
\operatorname{rg}(Q) + \dim(\ker(Q)) = 12
$$

Para que exista una única solución no nula salvo un factor
de escala, necesitamos que el núcleo de $Q$ tenga dimensión 1:

$$
\dim(\ker(Q)) = 1
$$

Esto significa que todas las soluciones pertenecen al
subespacio generado por un mismo vector no nulo $m_0$:

$$
m = \alpha m_0, \qquad \alpha \neq 0
$$

Reemplazando en el teorema de rango-nulidad:

$$
\operatorname{rg}(Q) + 1 = 12
$$

Por lo tanto:

$$
\boxed{\operatorname{rg}(Q)=11}
$$

De esta manera, el sistema tiene un núcleo unidimensional,
lo que garantiza que la matriz de proyección queda determinada
salvo un factor de escala.

**2. Cantidad mínima de puntos**

Cada correspondencia entre un punto tridimensional
$(U_i,V_i,W_i)$ y su proyección $(u_i,v_i)$ aporta dos
ecuaciones lineales al sistema.

Por lo tanto, para $n$ puntos, la matriz $Q$ tiene
$2n$ filas y 12 columnas.

Sabemos que el rango de una matriz está acotado por
su cantidad de filas y columnas:

$$
\operatorname{rg}(Q) \leq \min(2n,12)
$$

Como necesitamos que el rango sea 11:

$$
2n \geq 11
$$

Despejando:

$$
n \geq \frac{11}{2}
$$

Dado que la cantidad de puntos debe ser entera:

$$
\boxed{n_{\min}=6}
$$

Por lo tanto, necesitamos al menos 6 correspondencias
entre puntos 3D y puntos 2D.

**3. Condiciones que deben cumplir los puntos**

Tener 6 puntos es una condición necesaria, pero no
suficiente para garantizar la solución única.

Los puntos deben encontrarse en una configuración
no degenerada, de modo que las ecuaciones aporten
suficiente información independiente.

En particular, los puntos tridimensionales no deben
ser todos coplanares, ya que en ese caso no es posible
determinar una matriz de proyección 3D general de forma única.

Además, deben evitarse otras configuraciones degeneradas
que provoquen dependencias entre las filas de $Q$.

La condición algebraica que garantiza la unicidad es:

$$
\boxed{\operatorname{rg}(Q)=11}
$$

**Conclusión**

Para que el sistema homogéneo $Qm=0$ tenga una solución
no nula única salvo un factor de escala, necesitamos
que el núcleo de $Q$ tenga dimensión 1.

Por el teorema de rango-nulidad, esto equivale a:

$$
\boxed{\operatorname{rg}(Q)=11}
$$

Como cada correspondencia aporta dos ecuaciones, se
requieren al menos 6 puntos en una configuración
no degenerada que permita alcanzar dicho rango.

### Ejercicio 1.b – Resolución del problema de minimización

Buscamos encontrar el vector $m$ que resuelve:

$$
\min_{\|m\|_2=1}m^TQ^TQm
$$

**1. Propiedades de la matriz $Q^TQ$**

Definimos $A=Q^TQ$. Primero, observamos que $A$ es simétrica, ya que:

$$
A^T=(Q^TQ)^T=Q^T(Q^T)^T=Q^TQ=A
$$

Además, $A$ es semidefinida positiva porque, para cualquier vector $x\in\mathbb{R}^{12}$:

$$
x^TAx=x^TQ^TQx=(Qx)^T(Qx)=\|Qx\|_2^2\geq0
$$

Por lo tanto, todos los autovalores de $A$ son reales y no negativos.

**2. Aplicación del teorema espectral**

Como $A$ es simétrica, por el teorema espectral existe una base ortonormal de autovectores $v_1,\ldots,v_{12}$, asociados a los autovalores:

$$
\lambda_1\leq\lambda_2\leq\cdots\leq\lambda_{12}
$$

Podemos expresar cualquier vector $m$ como:

$$
m=\sum_{i=1}^{12}c_iv_i
$$

Como los autovectores son ortonormales y $\|m\|_2=1$, se cumple:

$$
\sum_{i=1}^{12}c_i^2=1
$$

**3. Determinación del mínimo**

Reemplazando la expresión de $m$ en la función que queremos minimizar y utilizando que $Av_i=\lambda_iv_i$ y $v_i^Tv_j=0$ para $i\neq j$, obtenemos:

$$
m^TAm=\sum_{i=1}^{12}\lambda_ic_i^2
$$

Como $\lambda_1$ es el menor autovalor y $c_i^2\geq0$:

$$
m^TAm=\sum_{i=1}^{12}\lambda_ic_i^2
\geq\lambda_1\sum_{i=1}^{12}c_i^2
=\lambda_1
$$

Esta cota se alcanza tomando $m=v_1$, ya que $v_1$ es un autovector unitario asociado a $\lambda_1$.

Por lo tanto:

$$
\boxed{
\min_{\|m\|_2=1}m^TQ^TQm
=\lambda_{\min}(Q^TQ)
}
$$

**Conclusión**

El vector $m$ que resuelve el problema de minimización es un autovector unitario de $Q^TQ$ asociado a su menor autovalor:

$$
\boxed{Q^TQm=\lambda_{\min}m}
$$


### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Para cada par de puntos $p_i$ y $P_i$, construimos una matriz $\hat{Q}$ . Agregamos un 1 al final  cada  $p_i$ en R2 y $P_i$ R3 :

$$\begin{pmatrix}u\\v\\1\end{pmatrix}(U\quad V\quad W\quad 1)=\begin{pmatrix}uU&uV&uW&u\\vU&vV&vW&v\\U&V&W&1\end{pmatrix}$$

Defino una matriz que me permita permutar y cambiar de signo
$$\begin{pmatrix}0&0&1\\-1&0&0\\0&-1&0\end{pmatrix}\quad \hat{Q}=\begin{pmatrix}U&V&W&1\\-uU&-uV&-uW&-u\\-vU&-vV&-vW&-v\end{pmatrix}$$

Defino un transformación lineal $T:\mathbb{R}^{3\times4}\rightarrow\mathbb{R}^{2\times12}$

$$\begin{pmatrix}m_{11}&m_{12}&m_{13}&m_{14}\\m_{21}&m_{22}&m_{23}&m_{24}\\m_{31}&m_{32}&m_{33}&m_{34}\end{pmatrix}=\begin{pmatrix}m_{11}&m_{12}&m_{13}&1&0&0&0&0&m_{21}&m_{22}&m_{23}&m_{24}\\0&0&0&0&m_{11}&m_{12}&m_{13}&1&m_{31}&m_{32}&m_{33}&m_{34}\end{pmatrix}$$

Matriz asociada a $T$

$$[T]=\begin{pmatrix}[T(E_1)]&[T(E_2)]&\cdots&[T(E_{12})]\end{pmatrix}\in\mathbb{R}^{24\times12}$$


Transformamos la matriz $\hat{Q}\in\mathbb{R}^{3\times4}$ en un vector columna $q\in\mathbb{R}^{12\times1}$, conservando sus elementos y ordenándolos por filas.

Multiplico ${T}$ ${q}$ = $$\begin{pmatrix}U_i&V_i&W_i&1&0&0&0&0&-u_iU_i&-u_iV_i&-u_iW_i&-u_i\\0&0&0&0&U_i&V_i&W_i&1&-v_iU_i&-v_iV_i&-v_iW_i&-v_i\end{pmatrix}$$

###b) Obtención de la matriz de proyección

En este ejercicio buscamos obtener la matriz de proyección $M$ a partir de la matriz $Q$ construida mediante el método DLT.

Como demostramos en el **Ejercicio 1.b**, el vector $m$ que minimiza:

$$
\min_{\|m\|_2=1}m^TQ^TQm
$$

es un autovector unitario asociado al menor autovalor de $Q^TQ$.

Por lo tanto, primero calculamos:

$$
A=Q^TQ
$$

Luego, obtenemos el autovector $m$ correspondiente al menor autovalor de $A$:

$$
Am=\lambda_{\min}m
$$

Una vez obtenido $m\in\mathbb{R}^{12}$, reorganizamos sus componentes para construir la matriz de proyección de dimensión $3\times4$:

$$
M=
\begin{pmatrix}
m_1&m_2&m_3&m_4\\
m_5&m_6&m_7&m_8\\
m_9&m_{10}&m_{11}&m_{12}
\end{pmatrix}
$$

**Conclusión**

De esta manera, obtenemos la matriz de proyección $M$ aplicando el resultado demostrado en el Ejercicio 1.b.

Cabe destacar que $M$ queda determinada salvo un factor de escala no nulo, ya que multiplicarla por un escalar no modifica la proyección representada.

### c) Recuperación de las matrices de calibración

Buscamos recuperar la matriz intrínseca $K$, la matriz de rotación $R$
y el vector de traslación $t$, a partir de la matriz de proyección $M$.

Sabemos que:

$$
M = \lambda K[R \mid t]
$$

Primero dividimos $M$ en una matriz $A$ de dimensión $3 \times 3$
y un vector $b$ de dimensión 3:

$$
M = [A \mid b]
$$

Como necesitamos que $K$ sea triangular superior y $R$ ortogonal,
aplicamos la factorización QR a la inversa de $A$:

$$
A^{-1} = Q_1 R_1
$$

Despejando:

$$
A = R_1^{-1}Q_1^T
$$

Por lo tanto, obtenemos las matrices provisionales:

$$
K_0 = R_1^{-1}, \qquad R_0 = Q_1^T
$$

Normalizamos $K_0$ para garantizar que $K_{33}=1$:

$$
c = (K_0)_{33}, \qquad K = \frac{K_0}{c}
$$

Como la matriz de rotación debe tener determinante 1,
definimos:

$$
s = \operatorname{sign}(\det(R_0))
$$

Y calculamos:

$$
R = sR_0
$$

El factor de escala de la proyección es:

$$
\lambda = cs
$$

Finalmente, como $b = \lambda Kt$, despejamos la traslación:

$$
t = \frac{1}{\lambda}K^{-1}b
$$

Reemplazando las expresiones anteriores y simplificando:

$$
t = sR_1b
$$

De esta manera recuperamos las matrices de calibración $K$, $R$
y el vector de traslación $t$.

d)


### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


- ¿Qué rango debe tener $Q\in\mathbb{R}^{2n\times9}$ para asegurar una solución única de $H$, a menos de un factor de escala?

**Por teorema Rango-nulidad**

Sea $M$ una matriz de dimensión $m\times n$, con $n$ columnas. Entonces:

$$\operatorname{rango}(M)+\dim(\operatorname{Nucleo}(M))=n$$

Sea $Q\in\mathbb{R}^{2n\times9}$.

Como queremos una solución única para $Qx=0$, salvo un factor de escala, necesitamos que:

$$\dim(\operatorname{Nucleo}(Q))=1$$

Por lo tanto:

$$\operatorname{rango}(Q)=9-\dim(\operatorname{Nucleo}(Q))=9-1=8$$

a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

Para obtener los puntos de las imagenes usamos estas funciones para crear un txt con la misma estructura que el archivo detected_corners.txt

In [2]:
def obtener_imgPts(ruta_imagen):
    """Detecta las 77 esquinas internas y devuelve sus coordenadas (u,v)."""
    imagen = cv2.imread(str(ruta_imagen))
    if imagen is None:
        raise FileNotFoundError(f"No se pudo leer: {ruta_imagen}")
    gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
    encontrado, esquinas = cv2.findChessboardCorners(gris, (11, 7))
    if not encontrado:
        raise ValueError(f"No se detectaron las esquinas: {ruta_imagen.name}")
    criterio = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    esquinas = cv2.cornerSubPix(gris, esquinas, (11, 11), (-1, -1), criterio)
    return esquinas.reshape(77, 2)

def guardar_imagen_puntos(ruta_imagen, imgPts, carpeta_salida):
    """Guarda la imagen con las esquinas detectadas y numeradas."""
    imagen = cv2.imread(str(ruta_imagen))
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    alto, ancho = imagen.shape[:2]
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(imagen)
    ax.scatter(imgPts[:, 0], imgPts[:, 1], color="red", s=12)
    for i in range(len(imgPts)):
        ax.text(imgPts[i, 0] + 3, imgPts[i, 1] - 5, str(i), color="blue", fontsize=6)
    ax.set_xlim(0, ancho)
    ax.set_ylim(alto, 0)
    ax.set_title(ruta_imagen.name)
    plt.tight_layout()
    plt.savefig(carpeta_salida / ruta_imagen.name, dpi=180)
    plt.close(fig)

def procesar_imagenes(ruta_carpeta):
    """Procesa las 20 imágenes, genera un TXT y guarda las imágenes numeradas."""
    carpeta = Path(ruta_carpeta)
    if not carpeta.is_dir():
        raise FileNotFoundError(f"No existe la carpeta: {carpeta}")
    carpeta_salida = carpeta / "puntos_detectados"
    carpeta_salida.mkdir(exist_ok=True)
    ruta_txt = carpeta / "detected_corners_2d.txt"
    datos = []
    errores = []
    for numero in range(1, 21):
        ruta_imagen = carpeta / f"Im_R_{numero}.png"
        try:
            imgPts = obtener_imgPts(ruta_imagen)
            guardar_imagen_puntos(ruta_imagen, imgPts, carpeta_salida)
            datos.append(imgPts.reshape(-1))
            print(f"Imagen {numero}: OK - 77 puntos")
        except (FileNotFoundError, ValueError, cv2.error) as error:
            errores.append((numero, str(error)))
            print(f"Imagen {numero}: ERROR - {error}")
    if errores:
        print("No se generó el TXT porque hubo errores de detección.")
        return
    matriz_datos = np.array(datos)
    np.savetxt(ruta_txt, matriz_datos, fmt="%.6f", delimiter=",")
    print("Procesamiento finalizado")
    print("Dimensión:", matriz_datos.shape)
    print("TXT:", ruta_txt)
    print("Imágenes:", carpeta_salida)


In [3]:
detected_corners_2d=' calib/coplanar/detected_corners_2d.txt'
datos_pixels=np.loadtxt(detected_corners_2d,delimiter=',')
print(f'Cantidad de imagenes : {len(datos_pixels)}') 

Cantidad de imagenes : 20


Los datos U V los contruimos con la función construir_objPts.

Los puntos u v lso tomamos del TXT 'detected_corners_2d'

Obtenemos la matriz K y obtenenmos R y t para la imagen 1.


In [4]:
datos_puntos_r2=construir_objPts() #construyo los puntos 
lista_H=[]
for x in range(0,len(datos_pixels),1):
    Q=obtenerQ2d(datos_pixels[x],datos_puntos_r2)
    H=resolverQhb(Q)
    lista_H.append(H)

A=armarA(lista_H)
K=calcularMatrizK2D(A)
# Pose para las imagen 1 
R1,t1=obtenerPose2D(lista_H[0],K)
R20,t20=obtenerPose2D(lista_H[19],K)

print('K:\n')
print(f'{K}\n')
print('R para la imagen 1:\n')
print(f'{R1}\n')
print('t para la imagen 1:\n')
print(f'{t1}\n')
print('R para la imagen 20:\n')
print(f'{R20}\n')
print('t para la imagen 20:\n')
print(f'{t20}\n')



K:

[[745.22306356  -1.00241417 519.502041  ]
 [  0.         756.52766835 298.48824378]
 [  0.           0.           1.        ]]

R para la imagen 1:

[[-0.99954339  0.01258709 -0.02778787]
 [-0.01206969 -0.99951653  0.0079752 ]
 [-0.02770075  0.00832767  0.99921207]]

t para la imagen 1:

[ 108.62062315  141.27649317 -550.35908827]

R para la imagen 20:

[[-0.98991289 -0.02645985 -0.13910929]
 [ 0.03880766 -0.99546121 -0.08488287]
 [-0.13625873 -0.08938994  0.98644673]]

t para la imagen 20:

[  83.22848026  146.30289187 -489.26887681]



Calculamos H y K con las funciones de OpenCV

In [5]:
# 1. Construcción de puntos físicos
puntos_objeto = construir_objPts().astype(np.float64)

# 2. Homografías con OpenCV
lista_H_cv = []
for imagen in datos_pixels:
    puntos_imagen = imagen.reshape(77, 2).astype(np.float64)
    H_cv, _ = cv2.findHomography(puntos_objeto, puntos_imagen, method=0)
    lista_H_cv.append(H_cv)

# 3. Comparación de homografías
H_propia = lista_H[0] / lista_H[0][2, 2]
H_cv = lista_H_cv[0] / lista_H_cv[0][2, 2]
print("H_propia - H_cv:")
print(np.round(H_propia - H_cv, 6))

# 4. Preparación de puntos para calibrateCamera
puntos_3d = np.zeros((77, 3), dtype=np.float32)
puntos_3d[:, :2] = puntos_objeto
objectPoints = [puntos_3d.copy() for _ in range(len(datos_pixels))]
imagePoints = [imagen.reshape(77, 2).astype(np.float32) for imagen in datos_pixels]

# 5. Calibración con OpenCV
ancho, alto = 1024, 576
error, K_cv, dist, rvecs, tvecs = cv2.calibrateCamera(objectPoints, imagePoints, (ancho, alto), None, None)

# 6. Comparación 
print("\nK propia:")
print(K)
print("\nK OpenCV:")
print(K_cv)
print("\nDiferencia K:")
print(np.round(K - K_cv, 6))


H_propia - H_cv:
[[-2.400e-05 -3.200e-05  1.917e-03]
 [-7.000e-06 -2.100e-05  9.230e-04]
 [-0.000e+00 -0.000e+00  0.000e+00]]

K propia:
[[745.22306356  -1.00241417 519.502041  ]
 [  0.         756.52766835 298.48824378]
 [  0.           0.           1.        ]]

K OpenCV:
[[722.05964298   0.         513.69586441]
 [  0.         732.73467418 292.73935449]
 [  0.           0.           1.        ]]

Diferencia K:
[[23.163421 -1.002414  5.806177]
 [ 0.       23.792994  5.748889]
 [ 0.        0.        0.      ]]


Calculamos K con nuestras funciones a partir de las H calculadas por OpenCV

In [6]:
A_cv = armarA(lista_H_cv)
K_homografias_cv = calcularMatrizK2D(A_cv)

print("K propia:")
print(K)
print("\nK con homografías OpenCV:")
print(K_homografias_cv)
print("\nDiferencia:")
print(np.round(K - K_homografias_cv, 6))

K propia:
[[745.22306356  -1.00241417 519.502041  ]
 [  0.         756.52766835 298.48824378]
 [  0.           0.           1.        ]]

K con homografías OpenCV:
[[744.49492951  -1.02581614 519.77416871]
 [  0.         755.78941267 298.56628446]
 [  0.           0.           1.        ]]

Diferencia:
[[ 0.728134  0.023402 -0.272128]
 [ 0.        0.738256 -0.078041]
 [ 0.        0.        0.      ]]


Calculamos R y t para la imagen 1 y comparamos vs los resultados de OpenCV

In [7]:
H = lista_H[0]
R_propia, t_propia = obtenerPose2D(H, K)
puntos_3d = np.zeros((77, 3), dtype=np.float64)
puntos_3d[:, :2] = datos_puntos_r2
puntos_2d = datos_pixels[0].reshape(77, 2).astype(np.float64)
dist = np.zeros(5)
exito, rvec, t_cv = cv2.solvePnP(puntos_3d, puntos_2d, K, dist, flags=cv2.SOLVEPNP_ITERATIVE)
R_cv, _ = cv2.Rodrigues(rvec)
print("R propia:")
print(R_propia)
print("\nR OpenCV:")
print(R_cv)
print("\nt propia:")
print(t_propia)
print("\nt OpenCV:")
print(t_cv.reshape(3))
print("\nDiferencia R:")
print(np.round(R_propia - R_cv, 6))
print("\nDiferencia t:")
print(np.round(t_propia - t_cv.reshape(3), 6))

R propia:
[[-0.99954339  0.01258709 -0.02778787]
 [-0.01206969 -0.99951653  0.0079752 ]
 [-0.02770075  0.00832767  0.99921207]]

R OpenCV:
[[ 0.99958029 -0.01254619 -0.02611203]
 [ 0.0126982   0.99990333  0.00566366]
 [ 0.02603845 -0.00599286  0.99964298]]

t propia:
[ 108.62062315  141.27649317 -550.35908827]

t OpenCV:
[-108.57904905 -141.40488469  550.55988548]

Diferencia R:
[[-1.999124e+00  2.513300e-02 -1.676000e-03]
 [-2.476800e-02 -1.999420e+00  2.312000e-03]
 [-5.373900e-02  1.432100e-02 -4.310000e-04]]

Diferencia t:
[  217.199672   282.681378 -1100.918974]


### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo.
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$.
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$.

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---